# Debug List Scraper NusaDaily
List-only. Tidak scrape artikel detail.

In [1]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.nusadaily as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/nusadaily.py


In [2]:
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [3]:
def debug_scrape_list(cutoff, max_pages=MAX_PAGES):
    rows = []
    for page_num in range(1, max_pages + 1):
        url = f'{scraper.BASE_URL}?q=kabupaten+malang&page={page_num}'
        print(f'Scraping NusaDaily page {page_num}')
        try:
            soup = fetch_html(url)
        except Exception as error:
            print(f'Gagal buka NusaDaily page {page_num}: {error}')
            break

        cards = soup.select('div.post-item')
        if not cards:
            break

        stop = False
        for card in cards:
            title_tag = card.select_one('h3.title a')
            date_tag = card.select_one('p.post-meta span')
            if not title_tag:
                continue
            published_date = date_tag.get_text(strip=True) if date_tag else None
            if is_older_than_cutoff(published_date, cutoff):
                stop = True
                break
            rows.append({
                'page_num': page_num,
                'list_page_url': url,
                'title': title_tag.get_text(strip=True),
                'url': title_tag['href'],
                'published_date': published_date,
            })
        if stop:
            break
    return records_to_df(rows)

list_df = debug_scrape_list(cutoff)
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)


Scraping NusaDaily page 1
page=001 | date=Jun 25, 2026 | title=Mengaku Pegawai Pemprov Jatim Tawarkan Program UMKM Fik...
page=001 | date=Jun 25, 2026 | title=PDIP Kabupaten Malang Tutup Bulan Bung Karno dengan Taw...
page=001 | date=Jun 13, 2026 | title=Imigrasi Malang Resmikan Bantur Kabupaten Malang sebaga...
page=001 | date=Jun 5, 2026 | title=22 SPPG di Kabupaten Malang Disuspend Karena Tak Miliki...
page=001 | date=Jun 3, 2026 | title=Fraksi Gerindra Kabupaten Malang Dukung Langkah Tegas P...
page=001 | date=May 25, 2026 | title=Program BMW Bapenda Pertama di Jatim, Dorong Peningkata...
page=001 | date=May 20, 2026 | title=Kejari Kabupaten Malang Musnahkan Ratusan Gram Sabu, Ri...
page=001 | date=May 14, 2026 | title=Pasca Kunker Wabup Malang Bertemu Wapres, DPRD Kabupate...
page=001 | date=May 7, 2026 | title=Alun-Alun Kabupaten Malang Segera Terwujud, Pembangunan...
page=001 | date=May 6, 2026 | title=Fakta Baru Insiden Pantai Wediawu, 31 Wisatawan Asal Su...
page=001 | date=Ap

In [4]:
list_df = audit_list(list_df)


total rows: 15
first page: 1
last page: 1
newest date: 2026-06-25 00:00:00
oldest date: 2026-03-12 00:00:00
cutoff date: 2026-02-28 08:58:22.344678
reached cutoff: False
null parsed dates: 0

rows per month:


,month,count
0,2026-03,3
1,2026-04,2
2,2026-05,5
3,2026-06,5



rows per page:


,page_num,rows,newest,oldest
0,1,15,2026-06-25,2026-03-12


In [6]:
output_path = PROJECT_ROOT / 'csv' / 'nusadaily_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/nusadaily_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'nusadaily_article_debug.csv'

article_rows = []
sample_rows = list_df.head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
